<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# %% [markdown]
# Build clean demo artefacts from extracted keyword runs
# -----------------------------------------------------
# Inputs (already produced by training notebooks):
#   /content/drive/MyDrive/gt-markets/app-demo/extracted/{daily|weekly}/*.csv
#     - contain Date and either position OR prob_up (validation/test folds)
#     - usually *no* Close column -> I will load prices from processed/baseline.
#
# Outputs (what the Streamlit demo app expects to exist):
#   AppDemo/artefacts/{ASSET}/
#     ├─ metrics_baseline_D.csv  (pre-existing from baseline notebook)
#     ├─ metrics_baseline_W.csv
#     ├─ metrics_keywords_D.csv  (built here, clean, deduped)
#     ├─ metrics_keywords_W.csv
#     ├─ signals_<STRAT>_D.csv   (built here)
#     ├─ signals_<STRAT>_W.csv   (built here)
#     └─ figs/<STRAT>_<FREQ>.png (optional chart for quick sanity check)
#
# Notes for my future self:
# - I accept files like "*val_f0.csv", "*test_f0.csv" etc. They often lack Close.
# - For Close I first try data/processed, then any baseline signals_* for the asset/freq.
# - I align indices to Date everywhere to avoid pandas warnings.
# - Strategy names: KW_<MODEL>[_wNN][_<BASE|EXT>] so the app dropdown looks clean.
# - If duplicates exist for (asset,freq,strategy), I keep the one with best Sharpe.

# %%
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, math
from datetime import datetime

# --- Project layout (keep these paths consistent)
PROJECT_DIR    = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT    = PROJECT_DIR / "app-demo" / "extracted"     # has 'daily' and 'weekly'
ARTE_ROOT      = PROJECT_DIR / "AppDemo" / "artefacts"      # app reads from here
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"         # canonical prices live here (from main pipeline)

ASSETS         = ["GOLD","BTC","OIL","USDCNY"]
FREQ_DIRS      = {"daily":"D", "weekly":"W"}
ANNUALIZATION  = {"D":252, "W":52}
COST_BPS       = 5  # one-way cost for position flips

# Make sure folders exist
ARTE_ROOT.mkdir(parents=True, exist_ok=True)
for a in ASSETS:
    (ARTE_ROOT/a).mkdir(parents=True, exist_ok=True)
    (ARTE_ROOT/a/"figs").mkdir(parents=True, exist_ok=True)

# ------------------------
# Small utilities I keep reusing
# ------------------------

def _infer_asset_from_text(text: str):
    t = text.lower()
    if ("gold" in t) or ("gc=f" in t):   return "GOLD"
    if ("btc"  in t) or ("bitcoin" in t):return "BTC"
    if ("oil"  in t) or ("cl=f" in t):   return "OIL"
    if "usdcny" in t:                    return "USDCNY"
    return None

def _infer_model_from_text(text: str):
    t = text.lower()
    if "xgb" in t or "xgboost" in t: return "XGB"
    if "lstm" in t:                  return "LSTM"
    if "gru" in t:                   return "GRU"
    if "rf" in t or "randomforest" in t: return "RF"
    if re.search(r"(^|[^a-z])lr($|[^a-z])", t): return "LR"
    if "svm" in t: return "SVM"
    if "mlp" in t: return "MLP"
    return "KW"

def _infer_window_from_text(text: str):
    m = re.search(r"w(\d{1,3})", text.lower())
    return f"w{m.group(1)}" if m else None

def _infer_dataset_tag(text: str):
    t = text.lower()
    if "eng_ext" in t or "_ext" in t or "raw_ext" in t:   return "EXT"
    if "eng_base" in t or "_base" in t or "raw_base" in t: return "BASE"
    return None

def _compose_strategy(csv_path: Path):
    # Build consistent strategy label used by the app
    model = _infer_model_from_text(csv_path.name)
    win   = _infer_window_from_text(csv_path.name)
    tag   = _infer_dataset_tag(csv_path.as_posix())
    parts = [f"KW_{model}"]
    if win: parts.append(win)
    if tag: parts.append(tag)
    return "_".join(parts)

def _choose_asset_for_df(csv_path: Path, df: pd.DataFrame):
    # Prefer hints from the file/folder path, then fallback to columns if present
    a = _infer_asset_from_text(csv_path.as_posix())
    if a: return a
    for col in ["asset","symbol","ticker"]:
        if col in df.columns:
            aa = _infer_asset_from_text(str(df[col].iloc[0]))
            if aa: return aa
    return None

def _detect_position_column(df: pd.DataFrame):
    # Be liberal about column names used by the training notebook
    # Return series of 0/1 integers aligned to df index
    lower = {c.lower(): c for c in df.columns}
    if "position" in lower:
        s = pd.to_numeric(df[lower["position"]], errors="coerce").fillna(0.0)
        return (s >= 0.5).astype(int)
    for cand in ["prob_up","proba_up","p_up","prob","y_pred_proba","pred_proba","probability"]:
        if cand in lower:
            s = pd.to_numeric(df[lower[cand]], errors="coerce").fillna(0.0)
            return (s >= 0.5).astype(int)
    for cand in ["signal","y_pred","pred"]:
        if cand in lower:
            s = pd.to_numeric(df[lower[cand]], errors="coerce").fillna(0.0)
            # treat negative/zero as flat, positive as long
            return (s > 0).astype(int)
    return None  # nothing usable

def _load_processed_price(asset: str, F: str):
    """
    Try to load Close from data/processed. I allow a few common file name patterns
    produced by my pipeline. Returns pd.Series indexed by Date or None.
    """
    candidates = [
        DATA_PROCESSED / f"{asset.lower()}_{F}.csv",            # e.g., gold_D.csv
        DATA_PROCESSED / f"{asset}_{F}.csv",                    # e.g., GOLD_D.csv
        DATA_PROCESSED / f"prices_{asset}_{F}.csv",             # e.g., prices_GOLD_D.csv
        DATA_PROCESSED / f"{asset.lower()}_{F.lower()}.csv",    # e.g., gold_d.csv
    ]
    for p in candidates:
        if p.exists():
            try:
                d = pd.read_csv(p)
                # Figure out close column
                close_col = None
                for cand in ["Close","close","Adj Close","adjclose","price","Price"]:
                    if cand in d.columns: close_col = cand; break
                if ("Date" in d.columns) and close_col:
                    dates = pd.to_datetime(d["Date"], errors="coerce")
                    close = pd.to_numeric(d[close_col], errors="coerce")
                    s = pd.Series(close.values, index=dates, name="Close").dropna()
                    s.index = pd.to_datetime(s.index)
                    s = s.sort_index()
                    return s
            except Exception:
                pass
    return None

def _load_close_from_baseline_signals(asset: str, F: str):
    """
    As a fallback, grab any signals_*_{F}.csv from artefacts and steal the Close series.
    """
    base_dir = ARTE_ROOT / asset
    if not base_dir.exists(): return None
    for p in sorted(base_dir.glob(f"signals_*_{F}.csv")):
        try:
            d = pd.read_csv(p)
            if "Date" in d.columns and "Close" in d.columns:
                dates = pd.to_datetime(d["Date"], errors="coerce")
                close = pd.to_numeric(d["Close"], errors="coerce")
                s = pd.Series(close.values, index=dates, name="Close").dropna()
                s.index = pd.to_datetime(s.index)
                s = s.sort_index()
                return s
        except Exception:
            continue
    return None

def _get_close_series(asset: str, F: str):
    """
    Unified price loader. I first try processed data; if not found,
    I fallback to any existing baseline signals in artefacts.
    """
    s = _load_processed_price(asset, F)
    if s is not None: return s
    s = _load_close_from_baseline_signals(asset, F)
    return s  # may be None

def _to_equity(close: pd.Series, pos01: pd.Series, freq: str, cost_bps: float):
    # enforce datetime indices + alignment
    close = close.copy()
    close.index = pd.to_datetime(close.index, errors="coerce")
    close = close.dropna().sort_index()

    pos01 = pos01.copy().astype(float)
    pos01.index = pd.to_datetime(pos01.index, errors="coerce")
    pos01 = pos01.sort_index()
    pos01 = pos01.reindex(close.index).ffill().fillna(0.0).clip(0,1).astype(int)

    # Returns with no implicit padding => no FutureWarning
    ret = close.pct_change(fill_method=None).fillna(0.0)

    trans = pos01.diff().abs().fillna(0.0)             # 0/1 when switching
    strat = (pos01.shift(1).fillna(0.0) * ret) - trans * (cost_bps / 1e4)

    eq = (1.0 + strat).cumprod().replace([np.inf, -np.inf], np.nan).fillna(1.0)
    ann = ANNUALIZATION[freq]
    mu  = strat.mean() * ann
    sd0 = strat.std(ddof=0)
    sd  = sd0 * (ann ** 0.5) if sd0 > 0 else np.nan
    shr = (mu / sd) if (sd and sd > 0) else np.nan
    mdd = (eq / eq.cummax() - 1.0).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": shr, "MaxDD": mdd}

# ------------------------
# Main build loop
# ------------------------

rows_metrics = []
built_signals = []

for subdir, F in FREQ_DIRS.items():
    src = SRC_EXTRACT / subdir
    if not src.exists():
        print(f"[skip] no folder: {src}")
        continue

    # I process newest files last so "best Sharpe wins" de-dupe still keeps a stable pick.
    csvs = sorted(src.rglob("*.csv"), key=lambda p: p.stat().st_mtime)

    for csv in csvs:
        name = csv.name.lower()
        # Hard skip: non-signal tables I don't want in the app
        if any(x in name for x in ["leaderboard","scores","eval","backtest_summary"]):
            continue

        try:
            df = pd.read_csv(csv)
        except Exception as e:
            print(f"[warn] cannot read {csv.name}: {e}")
            continue

        # Standardize Date
        if "Date" not in df.columns:
            if "date" in df.columns:
                df = df.rename(columns={"date":"Date"})
            else:
                continue
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])

        # Asset + strategy labels
        asset = _choose_asset_for_df(csv, df)
        if asset not in ASSETS:
            # keep demo to the 4 focus assets
            continue
        strategy = _compose_strategy(csv)

        # Build binary position series
        pos01 = _detect_position_column(df)
        if pos01 is None:
            # nothing tradable in this file
            continue
        pos01.index = df["Date"]

        # Close: either present in df OR I fetch from processed/baseline
        close_series = None
        if "Close" in df.columns:
            close_series = pd.Series(pd.to_numeric(df["Close"], errors="coerce").values,
                                     index=df["Date"], name="Close")
        else:
            close_series = _get_close_series(asset, F)
            if close_series is None:
                print(f"[warn] missing Close and no fallback for {csv.name}; skipped")
                continue

        # Compute equity + metrics
        eq, mets = _to_equity(close_series, pos01, F, COST_BPS)

        # Persist signals file for the app
        out_dir = ARTE_ROOT / asset
        out_dir.mkdir(parents=True, exist_ok=True)

        # Signal convention for the app: NaN most of the time; 1 on entry, 0 on exit
        pos01_aligned = pos01.reindex(eq.index).ffill().fillna(0).astype(int)
        trans = pos01_aligned.diff().fillna(0).abs().astype(int)
        signal = trans.where(trans > 0, np.nan)
        signal = signal.where(signal.isna(), pos01_aligned)

        sig_df = pd.DataFrame({
            "Date": eq.index.strftime("%Y-%m-%d"),
            "signal": signal.values,
            "position": pos01_aligned.values,
            "Close": close_series.reindex(eq.index).ffill().values,
            "strategy": strategy
        })
        sig_path = out_dir / f"signals_{strategy}_{F}.csv"
        sig_df.to_csv(sig_path, index=False)
        built_signals.append((asset, F, strategy, sig_path))

        # Save a small equity curve figure (optional)
        fig_path = out_dir / "figs" / f"{strategy}_{F}.png"
        ax = eq.plot(figsize=(6,3), title=f"{asset} – {strategy} – {F}")
        ax.grid(alpha=0.3)
        ax.get_figure().savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(ax.get_figure())

        # Metrics row
        rows_metrics.append({
            "asset": asset, "freq": F, "strategy": strategy,
            **mets
        })

# ------------------------
# Aggregate and write metrics_keywords_*.csv per asset+freq
# ------------------------
if not rows_metrics:
    raise RuntimeError("No keyword strategies found after adding price fallbacks. Check processed prices under data/processed or baseline signals in artefacts.")

metrics_df = pd.DataFrame(rows_metrics).dropna(subset=["asset","freq","strategy"]).copy()

# De-duplication: best Sharpe per (asset,freq,strategy)
metrics_best = (
    metrics_df
    .sort_values(["asset","freq","strategy","Sharpe"], ascending=[True, True, True, False])
    .drop_duplicates(subset=["asset","freq","strategy"], keep="first")
    .reset_index(drop=True)
)

for a in ASSETS:
    for F in ["D","W"]:
        sub = metrics_best[(metrics_best.asset == a) & (metrics_best.freq == F)].copy()
        out_csv = ARTE_ROOT / a / f"metrics_keywords_{F}.csv"
        if len(sub) == 0:
            if out_csv.exists():
                out_csv.unlink()  # remove stale file so the app doesn't get confused
            continue
        sub[["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD"]].to_csv(out_csv, index=False)

# ------------------------
# Housekeeping: drop signals_* with no matching metrics entry (keeps repo tidy)
# ------------------------
valid = set((r.asset, r.freq, r.strategy) for r in metrics_best.itertuples(index=False))
removed = 0
for a in ASSETS:
    for p in (ARTE_ROOT/a).glob("signals_*.csv"):
        m = re.match(r"signals_(.+)_(D|W)\.csv$", p.name)
        if not m:
            continue
        strat, F = m.group(1), m.group(2)
        if (a, F, strat) not in valid:
            p.unlink(missing_ok=True)
            removed += 1

print(f"✅ Built keyword artefacts into: {ARTE_ROOT}")
if removed:
    print(f"🧹 Removed {removed} stale signal files without a metrics entry.")

# ------------------------
# Integrity check for the app
# ------------------------
_required = {
    "GOLD":  ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "BTC":   ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "OIL":   ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "USDCNY":["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
}
missing = []
for asset, files in _required.items():
    for f in files:
        if not (ARTE_ROOT/asset/f).exists():
            missing.append(f"{asset}/{f}")

if missing:
    print("⚠️ Missing artefact files:")
    for m in missing:
        print("  -", m)
else:
    print("✅ All required files present for the demo app.")


Mounted at /content/drive
✅ Built keyword artefacts into: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
🧹 Removed 112 stale signal files without a metrics entry.
✅ All required files present for the demo app.
